# Multilingual NER — BiLSTM-CRF Training

**Architecture:** Word Embedding (300d FastText) + CharCNN (50d) + LangID (16d) → BiLSTM (512d) → CRF

**Datasets:** CoNLL-2003 (EN) + WikiANN (EN/RU)

**Training strategy:**
1. Train on CoNLL-2003 English
2. Fine-tune on WikiANN Russian
3. Merged fine-tune on EN+RU combined

**Hardware:** Kaggle T4 x2 GPU

In [ ]:
!git clone https://github.com/uzbtrust/Uzbek-Operator-NER-From-Scratch.git
%cd Uzbek-Operator-NER-From-Scratch
!pip install -q datasets seqeval pyyaml

In [ ]:
import os
import sys
import time
import json
import yaml
from pathlib import Path

import torch
import torch.optim as optim
import numpy as np
from torch.utils.data import DataLoader, ConcatDataset
from torch.cuda.amp import GradScaler, autocast

sys.path.insert(0, ".")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")
if torch.cuda.is_available():
    for i in range(torch.cuda.device_count()):
        print(f"  GPU {i}: {torch.cuda.get_device_name(i)}")
    print(f"  Total GPUs: {torch.cuda.device_count()}")

## 1. Download Datasets

In [ ]:
from data.download_datasets import download_conll, download_wikiann

RAW_DIR = "data/raw"

download_conll(RAW_DIR)
download_wikiann("en", RAW_DIR)
download_wikiann("ru", RAW_DIR)

print("Datasets ready.")

## 2. Build Vocabularies

In [ ]:
from data.vocab import build_vocabs, Vocabulary, CharVocabulary, TagMap

PROC_DIR = "data/processed"

data_dirs = [
    Path(RAW_DIR) / "conll2003",
    Path(RAW_DIR) / "wikiann_en",
    Path(RAW_DIR) / "wikiann_ru",
]

word_vocab, char_vocab, tag_map = build_vocabs(data_dirs, PROC_DIR, min_freq=2)

print(f"Word vocab: {len(word_vocab)}")
print(f"Char vocab: {len(char_vocab)}")
print(f"Tags: {len(tag_map)} -> {list(tag_map.tag2idx.keys())}")

## 3. Download FastText Embeddings

In [ ]:
from embeddings.load_fasttext import download_vectors, load_vectors, build_embedding_matrix

VEC_DIR = "embeddings/vectors"
MAX_VECTORS = 500000

download_vectors("en", VEC_DIR)
download_vectors("ru", VEC_DIR)

In [ ]:
en_vecs = load_vectors(Path(VEC_DIR) / "cc.en.300.vec", MAX_VECTORS)
ru_vecs = load_vectors(Path(VEC_DIR) / "cc.ru.300.vec", MAX_VECTORS)

merged_vecs = {}
merged_vecs.update(en_vecs)
merged_vecs.update(ru_vecs)

pretrained_emb = build_embedding_matrix(word_vocab.word2idx, merged_vecs)
print(f"Embedding matrix: {pretrained_emb.shape}")

del en_vecs, ru_vecs, merged_vecs
torch.cuda.empty_cache()

## 4. Prepare Datasets

In [ ]:
from data.preprocess import NERDataset, collate_batch, load_raw_data

MAX_SEQ = 128
MAX_WORD = 30

def make_ds(name, split):
    samples = load_raw_data(Path(RAW_DIR) / name / f"{split}.json")
    return NERDataset(samples, word_vocab, char_vocab, tag_map, MAX_SEQ, MAX_WORD)

en_train = make_ds("conll2003", "train")
en_val   = make_ds("conll2003", "validation")
en_test  = make_ds("conll2003", "test")

wa_en_train = make_ds("wikiann_en", "train")
wa_en_val   = make_ds("wikiann_en", "validation")

ru_train = make_ds("wikiann_ru", "train")
ru_val   = make_ds("wikiann_ru", "validation")
ru_test  = make_ds("wikiann_ru", "test")

print(f"CoNLL EN  -> train: {len(en_train)}, val: {len(en_val)}, test: {len(en_test)}")
print(f"WikiANN EN -> train: {len(wa_en_train)}, val: {len(wa_en_val)}")
print(f"WikiANN RU -> train: {len(ru_train)}, val: {len(ru_val)}, test: {len(ru_test)}")

## 5. Build Model

In [ ]:
with open("configs/config.yaml", "r") as f:
    cfg = yaml.safe_load(f)

from model.ner_model import BiLSTMCRF

model = BiLSTMCRF(
    vocab_size=len(word_vocab),
    num_chars=len(char_vocab),
    num_tags=len(tag_map),
    word_dim=cfg["embeddings"]["word_dim"],
    char_dim=cfg["embeddings"]["char_dim"],
    char_filters=cfg["embeddings"]["char_filters"],
    char_kernel=cfg["embeddings"]["char_kernel_size"],
    num_langs=cfg["model"]["num_langs"],
    lang_dim=cfg["embeddings"]["lang_dim"],
    hidden_size=cfg["model"]["hidden_size"],
    num_layers=cfg["model"]["num_layers"],
    dropout=cfg["model"]["dropout"],
    pretrained_weights=pretrained_emb,
)
model.to(device)

total_p = sum(p.numel() for p in model.parameters())
trainable_p = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"Total params:     {total_p:,}")
print(f"Trainable params: {trainable_p:,}")

## 6. Training Utilities

In [ ]:
from training.evaluate import run_evaluation

CKPT_DIR = "checkpoints"
Path(CKPT_DIR).mkdir(exist_ok=True)


def save_ckpt(model, optimizer, scheduler, scaler, epoch, best_f1, path):
    torch.save({
        "epoch": epoch,
        "best_f1": best_f1,
        "model": model.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict() if scaler else None,
    }, path)


def train_stage(model, train_loader, val_loader, tag_map, lr, max_epochs, stage_name,
                grad_clip=5.0, patience=7, sched_patience=3, use_fp16=True):
    use_amp = use_fp16 and device.type == "cuda"
    optimizer = optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode="max", patience=sched_patience)
    scaler = GradScaler() if use_amp else None

    best_f1 = 0.0
    no_improve = 0
    history = []

    print(f"\n{'='*60}")
    print(f"  Stage: {stage_name} | lr={lr} | max_epochs={max_epochs}")
    print(f"{'='*60}")

    for epoch in range(max_epochs):
        model.train()
        total_loss, n_batch = 0, 0
        t0 = time.time()

        for batch in train_loader:
            w = batch["word_ids"].to(device)
            c = batch["char_ids"].to(device)
            t_ids = batch["tag_ids"].to(device)
            l = batch["lang_ids"].to(device)
            m = batch["mask"].to(device)
            lens = batch["lengths"].to(device)

            optimizer.zero_grad()

            if use_amp and scaler:
                with autocast():
                    loss = model(w, c, l, t_ids, m, lens)
                scaler.scale(loss).backward()
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = model(w, c, l, t_ids, m, lens)
                loss.backward()
                torch.nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
                optimizer.step()

            total_loss += loss.item()
            n_batch += 1

        avg_loss = total_loss / max(n_batch, 1)
        elapsed = time.time() - t0

        metrics = run_evaluation(model, val_loader, tag_map, device)
        val_f1 = metrics["overall"]["f1"]
        scheduler.step(val_f1)

        cur_lr = optimizer.param_groups[0]["lr"]
        history.append({"epoch": epoch+1, "loss": avg_loss, "val_f1": val_f1, "lr": cur_lr})

        marker = ""
        if val_f1 > best_f1:
            best_f1 = val_f1
            no_improve = 0
            save_ckpt(model, optimizer, scheduler, scaler, epoch, best_f1,
                     Path(CKPT_DIR) / f"{stage_name}_best.pt")
            marker = " *best*"
        else:
            no_improve += 1

        print(f"  Epoch {epoch+1:3d}/{max_epochs} | loss: {avg_loss:.4f} | "
              f"val_f1: {val_f1:.4f} | lr: {cur_lr:.2e} | {elapsed:.0f}s{marker}")

        if no_improve >= patience:
            print(f"  Early stopping at epoch {epoch+1}")
            break

    print(f"  Best F1: {best_f1:.4f}")
    return best_f1, history

## 7. Stage 1 — Train on CoNLL-2003 (English)

In [ ]:
BS = cfg["training"]["batch_size"]

en_train_loader = DataLoader(en_train, batch_size=BS, shuffle=True,
                             collate_fn=collate_batch, num_workers=2, pin_memory=True)
en_val_loader = DataLoader(en_val, batch_size=BS, shuffle=False,
                           collate_fn=collate_batch, num_workers=2, pin_memory=True)

en_f1, en_hist = train_stage(
    model, en_train_loader, en_val_loader, tag_map,
    lr=1e-3, max_epochs=50, stage_name="en_conll"
)

## 8. Stage 2 — Fine-tune on WikiANN Russian

In [ ]:
ru_train_loader = DataLoader(ru_train, batch_size=BS, shuffle=True,
                             collate_fn=collate_batch, num_workers=2, pin_memory=True)
ru_val_loader = DataLoader(ru_val, batch_size=BS, shuffle=False,
                           collate_fn=collate_batch, num_workers=2, pin_memory=True)

ru_f1, ru_hist = train_stage(
    model, ru_train_loader, ru_val_loader, tag_map,
    lr=1e-4, max_epochs=30, stage_name="ru_wikiann"
)

## 9. Stage 3 — Merged Fine-tune (EN + RU)

In [ ]:
merged_train = ConcatDataset([en_train, ru_train, wa_en_train])
merged_val = ConcatDataset([en_val, ru_val, wa_en_val])

merged_train_loader = DataLoader(merged_train, batch_size=BS, shuffle=True,
                                 collate_fn=collate_batch, num_workers=2, pin_memory=True)
merged_val_loader = DataLoader(merged_val, batch_size=BS, shuffle=False,
                               collate_fn=collate_batch, num_workers=2, pin_memory=True)

merged_f1, merged_hist = train_stage(
    model, merged_train_loader, merged_val_loader, tag_map,
    lr=1e-4, max_epochs=20, stage_name="merged"
)

## 10. Final Evaluation

In [ ]:
en_test_loader = DataLoader(en_test, batch_size=64, shuffle=False, collate_fn=collate_batch)
ru_test_loader = DataLoader(ru_test, batch_size=64, shuffle=False, collate_fn=collate_batch)

en_metrics = run_evaluation(model, en_test_loader, tag_map, device)
ru_metrics = run_evaluation(model, ru_test_loader, tag_map, device)

print("=" * 50)
print("FINAL TEST RESULTS")
print("=" * 50)
print(f"\nCoNLL-2003 English:")
print(f"  F1:        {en_metrics['overall']['f1']:.4f}")
print(f"  Precision: {en_metrics['overall']['precision']:.4f}")
print(f"  Recall:    {en_metrics['overall']['recall']:.4f}")
for ent, v in en_metrics["per_entity"].items():
    print(f"  {ent:8s}  P={v['precision']:.3f}  R={v['recall']:.3f}  F1={v['f1-score']:.3f}  ({v['support']})")

print(f"\nWikiANN Russian:")
print(f"  F1:        {ru_metrics['overall']['f1']:.4f}")
print(f"  Precision: {ru_metrics['overall']['precision']:.4f}")
print(f"  Recall:    {ru_metrics['overall']['recall']:.4f}")
for ent, v in ru_metrics["per_entity"].items():
    print(f"  {ent:8s}  P={v['precision']:.3f}  R={v['recall']:.3f}  F1={v['f1-score']:.3f}  ({v['support']})")

In [ ]:
results = {
    "stage_1_en_conll": {"best_f1": en_f1, "history": en_hist},
    "stage_2_ru_wikiann": {"best_f1": ru_f1, "history": ru_hist},
    "stage_3_merged": {"best_f1": merged_f1, "history": merged_hist},
    "test_en": en_metrics,
    "test_ru": ru_metrics,
}

Path("results").mkdir(exist_ok=True)
with open("results/training_results.json", "w") as f:
    json.dump(results, f, indent=2)

print("Results saved to results/training_results.json")

## 11. Training Curves

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, (name, hist) in zip(axes, [
    ("Stage 1: CoNLL EN", en_hist),
    ("Stage 2: WikiANN RU", ru_hist),
    ("Stage 3: Merged EN+RU", merged_hist),
]):
    epochs = [h["epoch"] for h in hist]
    losses = [h["loss"] for h in hist]
    f1s = [h["val_f1"] for h in hist]

    ax2 = ax.twinx()
    ax.plot(epochs, losses, "b-o", markersize=3, label="Loss")
    ax2.plot(epochs, f1s, "r-s", markersize=3, label="Val F1")

    ax.set_xlabel("Epoch")
    ax.set_ylabel("Loss", color="b")
    ax2.set_ylabel("F1", color="r")
    ax.set_title(name)
    ax.legend(loc="upper left")
    ax2.legend(loc="upper right")

plt.tight_layout()
plt.savefig("results/training_curves.png", dpi=150)
plt.show()
print("Saved to results/training_curves.png")

## 12. Quick Inference Test

In [ ]:
from training.predict import NERPredictor

predictor = NERPredictor("configs/config.yaml", "checkpoints/merged_best.pt", "data/processed")

test_sentences = [
    "John Smith works at Google in New York",
    "Phone: +998881234567, Address: Andijan region",
    "Компания Яндекс расположена в Москве",
    "Contact us at office@example.com or visit our Tashkent branch",
    "Рабочие часы: 9:00 - 18:00, понедельник - пятница",
]

for sent in test_sentences:
    print(f"\nInput: {sent}")
    entities = predictor.extract_entities(sent)
    if entities:
        for e in entities:
            print(f"  [{e['type']:4s}] {e['text']}")
    else:
        print("  (no entities)")

## 13. Save Final Model & Artifacts

In [ ]:
final_dir = Path("final_model")
final_dir.mkdir(exist_ok=True)

torch.save(model.state_dict(), final_dir / "model_weights.pt")

import shutil
shutil.copy("data/processed/word_vocab.json", final_dir / "word_vocab.json")
shutil.copy("data/processed/char_vocab.json", final_dir / "char_vocab.json")
shutil.copy("data/processed/tag_map.json", final_dir / "tag_map.json")
shutil.copy("configs/config.yaml", final_dir / "config.yaml")

print("Final model artifacts saved to final_model/")
for f in sorted(final_dir.iterdir()):
    size_mb = f.stat().st_size / (1024 * 1024)
    print(f"  {f.name}: {size_mb:.1f} MB")